# Data Pipeline — OrbitWatch
# 
# Input files required in data/raw/:
#   satellites_30days.txt  — satellite TLE data
#   debris_30days.txt      — debris TLE data
#
# Output files produced:
#   data/processed/satellites_parsed.parquet
#   data/processed/debris_parsed.parquet
#   data/processed/scaler_params.json
#   data/training/X.npy
#   data/training/y.npy
#
# Run all cells in order to regenerate training data.
# Estimated time: 1-2 hours on CPU.

In [ ]:
# STEP 1: Initialize project directory structure for data and models
import os
import sys
# Ensure we are in the project root
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
print(f'Current working directory: {os.getcwd()}')

import os
import json
import numpy as np
import pandas as pd
from sgp4.api import Satrec

# STEP 1 â€” CREATE FOLDER STRUCTURE
print('STEP 1: Creating folder structure...')
folders = [
    'data/raw',
    'data/processed',
    'data/training',
    'models',
    'backend',
    'notebooks'
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
print('Folder structure created.')

Current working directory: D:\spacex


STEP 1: Creating folder structure...
Folder structure created.


In [2]:
# STEP 2 â€” PARSE BOTH 3LE FILES
def parse_3le_robust(filepath, object_type):
    print(f'Parsing {filepath}...')
    records = []
    malformed_count = 0
    
    if not os.path.exists(filepath):
        print(f'  Warning: {filepath} not found.')
        return pd.DataFrame(), 0
        
    with open(filepath, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
        
    i = 0
    while i < len(lines):
        if lines[i].startswith('0 '):
            if i + 2 < len(lines) and lines[i+1].startswith('1 ') and lines[i+2].startswith('2 '):
                line0, line1, line2 = lines[i], lines[i+1], lines[i+2]
                try:
                    name = line0[2:].strip()
                    norad_id = int(line1[2:7])
                    sat = Satrec.twoline2rv(line1, line2)
                    epoch_jd = sat.jdsatepoch + sat.jdsatepochF
                    records.append({
                        'name': name,
                        'norad_id': norad_id,
                        'epoch_jd': epoch_jd,
                        'inclination': sat.inclo,
                        'eccentricity': sat.ecco,
                        'raan': sat.nodeo,
                        'arg_perigee': sat.argpo,
                        'mean_anomaly': sat.mo,
                        'mean_motion': sat.no_kozai,
                        'bstar': sat.bstar,
                        'object_type': object_type,
                        'line1': line1,
                        'line2': line2
                    })
                except Exception:
                    malformed_count += 1
                i += 3
            else:
                malformed_count += 1
                i += 1
        else:
            malformed_count += 1
            i += 1
            
    df = pd.DataFrame(records)
    print(f'  Parsed: {len(df)} records. Malformed: {malformed_count}')
    return df, malformed_count

print('\nSTEP 2: Parsing files...')
df_sat, err_sat = parse_3le_robust('data/raw/satellites_30days.txt', 'satellite')
df_deb, err_deb = parse_3le_robust('data/raw/debris_30days.txt', 'debris')

if df_sat.empty and df_deb.empty:
    df_all = pd.DataFrame()
    print('Warning: Both files are missing or empty. Pipeline might fail.')
else:
    df_all = pd.concat([df_sat, df_deb], ignore_index=True)
    print(f'Combined parser output: {len(df_all)} records.')


STEP 2: Parsing files...
Parsing data/raw/satellites_30days.txt...


  Parsed: 500000 records. Malformed: 0
Parsing data/raw/debris_30days.txt...


  Parsed: 500000 records. Malformed: 0
Combined parser output: 1000000 records.


In [3]:
# STEP 3 â€” DEDUPLICATE
if not df_all.empty:
    print('\nSTEP 3: Deduplicating...')
    initial_count = len(df_all)
    df_all = df_all.drop_duplicates(subset=['norad_id', 'epoch_jd'])
    df_all = df_all.sort_values(by=['norad_id', 'epoch_jd'])
    df_all = df_all.reset_index(drop=True)
    print(f'Records before: {initial_count}, Records after: {len(df_all)}')



STEP 3: Deduplicating...


Records before: 1000000, Records after: 862841


In [4]:
# STEP 4 â€” SGP4 PROPAGATION
if not df_all.empty:
    print('\nSTEP 4: SGP4 Propagation (this may take 10-20 minutes on CPU)...')
    x_km_list = []
    y_km_list = []
    z_km_list = []
    valid_mask = []
    
    total_rows = len(df_all)
    for idx, row in df_all.iterrows():
        if idx > 0 and idx % 50000 == 0:
            print(f'  Propagating row {idx}/{total_rows}...')
            
        sat = Satrec.twoline2rv(row['line1'], row['line2'])
        jd = int(row['epoch_jd'])
        fr = row['epoch_jd'] - jd
        
        e, r, v = sat.sgp4(jd, fr)
        if e == 0:
            x_km_list.append(r[0])
            y_km_list.append(r[1])
            z_km_list.append(r[2])
            valid_mask.append(True)
        else:
            x_km_list.append(np.nan)
            y_km_list.append(np.nan)
            z_km_list.append(np.nan)
            valid_mask.append(False)
            
    df_all['x_km'] = x_km_list
    df_all['y_km'] = y_km_list
    df_all['z_km'] = z_km_list
    
    survived = sum(valid_mask)
    print(f'  Rows survived propagation: {survived} out of {total_rows}')
    df_all = df_all[valid_mask].copy()
    
    df_all = df_all.drop(columns=['line1', 'line2'])
    df_all = df_all.reset_index(drop=True)



STEP 4: SGP4 Propagation (this may take 10-20 minutes on CPU)...


  Propagating row 50000/862841...


  Propagating row 100000/862841...


  Propagating row 150000/862841...


  Propagating row 200000/862841...


...

  Propagating row 700000/862841...


  Propagating row 750000/862841...


  Propagating row 800000/862841...


  Propagating row 850000/862841...


  Rows survived propagation: 862841 out of 862841


In [5]:
# STEP 5 â€” SAVE PARSED PARQUET FILES
if not df_all.empty:
    print('\nSTEP 5: Saving Parquet files...')
    df_sats = df_all[df_all['object_type'] == 'satellite']
    df_debs = df_all[df_all['object_type'] == 'debris']
    
    sats_path = 'data/processed/satellites_parsed.parquet'
    debs_path = 'data/processed/debris_parsed.parquet'
    
    df_sats.to_parquet(sats_path, engine='pyarrow')
    df_debs.to_parquet(debs_path, engine='pyarrow')
    
    print(f'  Saved {sats_path} - Size: {os.path.getsize(sats_path)/1024/1024:.2f} MB')
    print(f'  Saved {debs_path} - Size: {os.path.getsize(debs_path)/1024/1024:.2f} MB')



STEP 5: Saving Parquet files...


  Saved data/processed/satellites_parsed.parquet - Size: 35.14 MB
  Saved data/processed/debris_parsed.parquet - Size: 35.10 MB


In [6]:
# STEP 6 â€” FEATURE NORMALISATION
if not df_all.empty:
    print('\nSTEP 6: Feature Normalisation...')
    norm_cols = ['inclination', 'eccentricity', 'raan', 'arg_perigee', 'mean_anomaly', 'mean_motion', 'bstar', 'x_km', 'y_km', 'z_km']
    
    scaler_params = {}
    for col in norm_cols:
        mean_val = df_all[col].mean()
        std_val = df_all[col].std()
        
        if std_val == 0 or pd.isna(std_val):
            std_val = 1.0
            
        scaler_params[col] = {
            'mean': float(mean_val),
            'std': float(std_val)
        }
        
    with open('data/processed/scaler_params.json', 'w') as f:
        json.dump(scaler_params, f, indent=4)
        
    print('  Saved data/processed/scaler_params.json')
    
    for col in norm_cols:
        df_all[col] = (df_all[col] - scaler_params[col]['mean']) / scaler_params[col]['std']



STEP 6: Feature Normalisation...


  Saved data/processed/scaler_params.json


In [ ]:
# STEP 7 â€” BUILD SLIDING WINDOW SEQUENCES
if not df_all.empty:
    print('\nSTEP 7: Building Sliding Window Sequences...')
    FEATURE_COLS = ['inclination', 'eccentricity', 'raan', 'arg_perigee', 'mean_anomaly', 'mean_motion', 'bstar']
    TARGET_COLS = ['x_km', 'y_km', 'z_km']
    SEQ_LEN = 20
    PRED_STEP = 5
    MIN_RECORDS = SEQ_LEN + PRED_STEP + 1
    
    X_list = []
    y_list = []
    
    unique_ids = df_all['norad_id'].unique()
    for nid in unique_ids:
        group = df_all[df_all['norad_id'] == nid]
        if len(group) < MIN_RECORDS:
            continue
            
        group_features = group[FEATURE_COLS].values
        group_targets = group[TARGET_COLS].values
        
        n_rows = len(group)
        for i in range(n_rows - SEQ_LEN - PRED_STEP + 1):
            seq_x = group_features[i : i + SEQ_LEN]
            seq_y = group_targets[i + SEQ_LEN + PRED_STEP - 1]
            
            X_list.append(seq_x)
            y_list.append(seq_y)
            
    X_arr = np.array(X_list, dtype=np.float32)
    y_arr = np.array(y_list, dtype=np.float32)
    
    x_path = 'data/training/X.npy'
    y_path = 'data/training/y.npy'
    np.save(x_path, X_arr)
    np.save(y_path, y_arr)
    
    print(f'  Shape of X: {X_arr.shape}')
    print(f'  Shape of y: {y_arr.shape}')
    if len(X_arr) > 0:
        print(f'  Saved {x_path} - Size: {os.path.getsize(x_path)/1024/1024:.2f} MB')
        print(f'  Saved {y_path} - Size: {os.path.getsize(y_path)/1024/1024:.2f} MB')



STEP 7: Building Sliding Window Sequences...


  Shape of X: (547927, 20, 7)
  Shape of y: (547927, 3)
  Saved data/training/X.npy - Size: 292.62 MB
  Saved data/training/y.npy - Size: 6.27 MB


In [8]:
# STEP 8 â€” FINAL SUMMARY REPORT
if not df_all.empty:
    print('\nSTEP 8: Final Summary Report')
    print(f'Total raw records parsed (estimated combined): {initial_count + err_sat + err_deb}')
    print(f'Records after deduplication: {initial_count}')
    print(f'Records after propagation: {survived}')
    unique_sats = df_all[df_all['object_type']=='satellite']['norad_id'].nunique()
    unique_debs = df_all[df_all['object_type']=='debris']['norad_id'].nunique()
    print(f'Unique satellites processed: {unique_sats}')
    print(f'Unique debris objects processed: {unique_debs}')
    if len(X_arr) > 0:
        print(f'Total training sequences (X.npy rows): {len(X_arr)}')
        print(f'X.npy size in MB: {os.path.getsize(x_path)/1024/1024:.2f} MB')
        print(f'y.npy size in MB: {os.path.getsize(y_path)/1024/1024:.2f} MB')
    print('\nFiles saved:')
    print('  - data/processed/satellites_parsed.parquet')
    print('  - data/processed/debris_parsed.parquet')
    print('  - data/processed/scaler_params.json')
    if len(X_arr) > 0:
        print('  - data/training/X.npy')
        print('  - data/training/y.npy')



STEP 8: Final Summary Report
Total raw records parsed (estimated combined): 1000000
Records after deduplication: 1000000
Records after propagation: 862841


Unique satellites processed: 5308
Unique debris objects processed: 9242
Total training sequences (X.npy rows): 547927
X.npy size in MB: 292.62 MB
y.npy size in MB: 6.27 MB

Files saved:
  - data/processed/satellites_parsed.parquet
  - data/processed/debris_parsed.parquet
  - data/processed/scaler_params.json
  - data/training/X.npy
  - data/training/y.npy
